In [1]:
from docling.document_converter import DocumentConverter

converter = DocumentConverter()
document_source = "/Users/phung.pham/Documents/PHUNGPX/deepeval_exploration/data/papers/files/deepseek-ocrv2.pdf"
document = converter.convert(source=document_source).document

/Users/phung.pham/Documents/PHUNGPX/deepeval_exploration/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from docling_core.transforms.chunker.tokenizer.base import BaseTokenizer
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from transformers import AutoTokenizer

EMBED_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer: BaseTokenizer = HuggingFaceTokenizer(
    tokenizer=AutoTokenizer.from_pretrained(EMBED_MODEL_ID),
)

In [3]:
print(f"{tokenizer.get_max_tokens()=}")

tokenizer.get_max_tokens()=256


In [4]:
from typing import Iterable, Optional

from docling_core.transforms.chunker.base import BaseChunk
from docling_core.transforms.chunker.hierarchical_chunker import DocChunk
from docling_core.types.doc.labels import DocItemLabel
from rich.console import Console
from rich.panel import Panel


console = Console(width=200)

In [5]:
from docling_core.transforms.chunker.hierarchical_chunker import (
    ChunkingDocSerializer,
    ChunkingSerializerProvider,
    HierarchicalChunker,
)
from docling_core.transforms.serializer.markdown import MarkdownTableSerializer, MarkdownParams, MarkdownTextSerializer


class MDTableSerializerProvider(ChunkingSerializerProvider):
    def get_serializer(self, doc):
        return ChunkingDocSerializer(
            doc=doc,
            params=MarkdownParams(image_placeholder="<!-- image -->"),
            table_serializer=MarkdownTableSerializer(),  # configuring a different table serializer
            text_serializer=MarkdownTextSerializer(),
        )


chunker = HierarchicalChunker(
    serializer_provider=MDTableSerializerProvider(),
    merge_list_items=True,
)

chunk_iter = chunker.chunk(dl_doc=document)


def find_n_th_chunk_with_label(
    iter: Iterable[BaseChunk], n: int, label: DocItemLabel
) -> Optional[DocChunk]:
    num_found = -1
    for i, chunk in enumerate(iter):
        doc_chunk = DocChunk.model_validate(chunk)
        for it in doc_chunk.meta.doc_items:
            if it.label == label:
                num_found += 1
                if num_found == n:
                    return i, chunk
    return None, None


def print_chunk(chunks, chunk_pos):
    chunk = chunks[chunk_pos]
    ctx_text = chunker.contextualize(chunk=chunk)
    num_tokens = tokenizer.count_tokens(text=ctx_text)
    doc_items_refs = [it.self_ref for it in chunk.meta.doc_items]
    title = f"{chunk_pos=} {num_tokens=} {doc_items_refs=}"
    console.print(Panel(ctx_text, title=title))


chunks = list(chunk_iter)
i, chunk = find_n_th_chunk_with_label(chunks, n=0, label=DocItemLabel.TABLE)
if i is not None:
    print_chunk(
        chunks=chunks,
        chunk_pos=i,
    )

Token indices sequence length is longer than the specified maximum sequence length for this model (2029 > 512). Running this sequence through the model will result in indexing errors


╭───────────────────────────────────────────────────────────── chunk_pos=43 num_tokens=2029 doc_items_refs=['#/texts/189', '#/tables/1'] ──────────────────────────────────────────────────────────────╮
│ 4.2.3. Continue-training LLM                                                                                                                                                                         │
│ Table 1 | Comprehensive evaluation of document reading on OmniDocBench v1.5. V-token 𝑚𝑎𝑥 represents the maximum number of visual tokens used per page in this benchmark. R-order denotes reading     │
│ order. Except for DeepSeek OCR and DeepSeek OCR 2, all other model results in this table are sourced from the OmniDocBench repository.                                                               │
│                                                                                                                                                                                                      │
│ | Model                   | V-token 𝑚𝑎𝑥 ↓ Overall ↑ Text 𝐸𝑑𝑖𝑡 ↓ Formula 𝐶𝐷𝑀 ↑ Table 𝑇𝐸𝐷𝑠 ↑ Table 𝑇𝐸𝐷𝑆 𝑠 ↑ R-order 𝐸𝑑𝑖𝑡 ↑   | V-token 𝑚𝑎𝑥 ↓ Overall ↑ Text 𝐸𝑑𝑖𝑡 ↓ Formula 𝐶𝐷𝑀 ↑ Table 𝑇𝐸𝐷𝑠 ↑ Table    │
│ 𝑇𝐸𝐷𝑆 𝑠 ↑ R-order 𝐸𝑑𝑖𝑡 ↑   | V-token 𝑚𝑎𝑥 ↓ Overall ↑ Text 𝐸𝑑𝑖𝑡 ↓ Formula 𝐶𝐷𝑀 ↑ Table 𝑇𝐸𝐷𝑠 ↑ Table 𝑇𝐸𝐷𝑆 𝑠 ↑ R-order 𝐸𝑑𝑖𝑡 ↑   | V-token 𝑚𝑎𝑥 ↓ Overall ↑ Text 𝐸𝑑𝑖𝑡 ↓ Formula 𝐶𝐷𝑀 ↑ Table 𝑇𝐸𝐷𝑠 ↑ Table    │
│ 𝑇𝐸𝐷𝑆 𝑠 ↑ R-order 𝐸𝑑𝑖𝑡 ↑   | V-token 𝑚𝑎𝑥 ↓ Overall ↑ Text 𝐸𝑑𝑖𝑡 ↓ Formula 𝐶𝐷𝑀 ↑ Table 𝑇𝐸𝐷𝑠 ↑ Table 𝑇𝐸𝐷𝑆 𝑠 ↑ R-order 𝐸𝑑𝑖𝑡 ↑   | V-token 𝑚𝑎𝑥 ↓ Overall ↑ Text 𝐸𝑑𝑖𝑡 ↓ Formula 𝐶𝐷𝑀 ↑ Table 𝑇𝐸𝐷𝑠 ↑ Table    │
│ 𝑇𝐸𝐷𝑆 𝑠 ↑ R-order 𝐸𝑑𝑖𝑡 ↑   | V-token 𝑚𝑎𝑥 ↓ Overall ↑ Text 𝐸𝑑𝑖𝑡 ↓ Formula 𝐶𝐷𝑀 ↑ Table 𝑇𝐸𝐷𝑠 ↑ Table 𝑇𝐸𝐷𝑆 𝑠 ↑ R-order 𝐸𝑑𝑖𝑡 ↑   |                                                                         │
│ |-------------------------|------------------------------------------------------------------------------------------------|------------------------------------------------------------------------ │
│ ------------------------|------------------------------------------------------------------------------------------------|-------------------------------------------------------------------------- │
│ ----------------------|------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------- │
│ --------------------|------------------------------------------------------------------------------------------------|                                                                               │
│ | Pipline                 | Pipline                                                                                        | Pipline                                                                 │
│ | Pipline                                                                                        | Pipline                                                                                        |  │
│ Pipline                                                                                        | Pipline                                                                                        |    │
│ Pipline                                                                                        |                                                                                                     │
│ | Marker-1.8.2 [1]        | -                                                                                              | 71.30                                                                   │
│ | 0.206                                                                                          | 76.66                                                                                          |  │
│ 57.88                                                                                          | 71.17                                                                             

In [6]:
def _print_chunk(chunk, chunk_pos):
    ctx_text = chunker.contextualize(chunk=chunk)
    num_tokens = tokenizer.count_tokens(text=ctx_text)
    doc_items_refs = [it.self_ref for it in chunk.meta.doc_items]
    title = f"{chunk_pos=} {num_tokens=} {doc_items_refs=}"
    console.print(Panel(ctx_text, title=title))

chunk_id = 0
for chunk in chunker.chunk(dl_doc=document):
    _print_chunk(chunk, chunk_id)
    chunk_id += 1

╭────────────────────────────────────────────────────────────────────── chunk_pos=0 num_tokens=8 doc_items_refs=['#/pictures/0'] ──────────────────────────────────────────────────────────────────────╮
│ <!-- image -->                                                                                                                                                                                       │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────── chunk_pos=1 num_tokens=23 doc_items_refs=['#/texts/2'] ───────────────────────────────────────────────────────────────────────╮
│ DeepSeek-OCR 2: Visual Causal Flow                                                                                                                                                                   │
│ Haoran Wei, Yaofeng Sun, Yukun Li                                                                                                                                                                    │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────── chunk_pos=2 num_tokens=16 doc_items_refs=['#/texts/3'] ───────────────────────────────────────────────────────────────────────╮
│ DeepSeek-OCR 2: Visual Causal Flow                                                                                                                                                                   │
│ DeepSeek-AI                                                                                                                                                                                          │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────────────────────────── chunk_pos=3 num_tokens=248 doc_items_refs=['#/texts/5'] ───────────────────────────────────────────────────────────────────────╮
│ Abstract                                                                                                                                                                                             │
│ We present DeepSeek-OCR 2 to investigate the feasibility of a novel encoder-DeepEncoder V2-capable of dynamically reordering visual tokens upon image semantics. Conventional vision-language models │
│ (VLMs) invariably process visual tokens in a rigid raster-scan order (top-left to bottom-right) with fixed positional encoding when fed into LLMs. However, this contradicts human visual            │
│ perception, which follows flexible yet semantically coherent scanning patterns driven by inherent logical structures. Particularly for images with complex layouts, human vision exhibits            │
│ causally-informed sequential processing. Inspired by this cognitive mechanism, DeepEncoder V2 is designed to endow the encoder with causal reasoning capabilities, enabling it to intelligently      │
│ reorder visual tokens prior to LLM-based content interpretation. This work explores a novel paradigm: whether 2D image understanding can be effectively achieved through two-cascaded 1D causal      │
│ reasoning structures, thereby offering a new architectural approach with the potential to achieve genuine 2D reasoning. Codes and model weights are publicly accessible at                           │
│ http://github.com/deepseek-ai/DeepSeek-OCR-2 .                                                                                                                                                       │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────── chunk_pos=4 num_tokens=81 doc_items_refs=['#/texts/6', '#/pictures/1'] ───────────────────────────────────────────────────────────────╮
│ Abstract                                                                                                                                                                                             │
│ Figure 1 | We substitute the CLIP component in DeepEncoder with an LLM-style architecture. By customizing the attention mask, visual tokens utilize bidirectional attention while learnable queries  │
│ adopt causal attention. Each query token can thus attend to all visual tokens and preceding queries, allowing progressive causal reordering over visual information.                                 │
│                                                                                                                                                                                                      │
│ <!-- image -->                                                                                                                                                                                       │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────────────────────────────── chunk_pos=55 num_tokens=69 doc_items_refs=['#/texts/211'] ──────────────────────────────────────────────────────────────────────╮
│ 7. Conclusion                                                                                                                                                                                        │
│ While optical text reading, particularly document parsing, represents one of the most practical vision tasks in the LLM era, it constitutes only a small part of the broader visual understanding    │
│ landscape. Looking ahead, we will refine and adapt this architecture to more diverse scenarios, seeking deeper toward a more comprehensive vision of multimodal intelligence.                        │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ chunk_pos=56 num_tokens=1089 doc_items_refs=['#/texts/214', '#/texts/215', '#/texts/216', '#/texts/217', '#/texts/218', '#/texts/219', '#/texts/220', '#/texts/221', '#/texts/222', '#/texts/223', ─╮
│ References                                                                                                                                                                                           │
│ - [1] Marker. URL https://github.com/datalab-to/marker .                                                                                                                                             │
│ - [2] Nanonets-ocr-s, 2025. URL https://huggingface.co/nanonets/Nanonets-OCR-s .                                                                                                                     │
│ - [3] Ocrverse, 2025. URL https://github.com/DocTron-hub/OCRVerse .                                                                                                                                  │
│ - [4] Ocrflux, 2025. URL https://github.com/chatdoc-com/OCRFlux .                                                                                                                                    │
│ - [5] Adept. Fuyu-8b. https://huggingface.co/adept/fuyu-8b , 2023.                                                                                                                                   │
│ - [6] G. AI. Gemini 2.5-pro, 2025. URL https://gemini.google.com/ .                                                                                                                                  │
│ - [7] J. Bai, S. Bai, S. Yang, S. Wang, S. Tan, P. Wang, J. Lin, C. Zhou, and J. Zhou. Qwen-vl: A versatile vision-language model for understanding, localization, text reading, and beyond. arXiv   │
│ preprint arXiv:2308.12966, 2023.                                                                                                                                                                     │
│ - [8] S. Bai, Y. Cai, R. Chen, et al. Qwen3-vl technical report. arXiv preprint arXiv:2511.21631, 2025. URL https://arxiv.org/abs/2511.21631 .                                                       │
│ - [9] S. Bai, K. Chen, X. Liu, J. Wang, W. Ge, S. Song, K. Dang, P. Wang, S. Wang, J. Tang, H. Zhong, Y. Zhu, M. Yang, Z. Li, J. Wan, P. Wang, W. Ding, Z. Fu, Y. Xu, J. Ye, X. Zhang, T. Xie, Z.    │
│ Cheng, H. Zhang, Z. Yang, H. Xu, and J. Lin. Qwen2.5-vl technical report. arXiv preprint arXiv:2502.13923, 2025.                                                                                     │
│ - [10] N. Carion, F. Massa, G. Synnaeve, N. Usunier, A. Kirillov, and S. Zagoruyko. End-to-end object detection with transformers. In European conference on computer vision, pages 213-229.         │
│ Springer, 2020.                                                                                                                                                                                      │
│ - [11] J. Chen, L. Kong, H. Wei, C. Liu, Z. Ge, L. Zhao, J. Sun, C. Han, and X. Zhang. Onechart: Purify the chart structural extraction via one auxiliary token. In Proceedings of the 32nd          │
│ ACMInternational Conference on Multimedia, pages 147-155, 2024.                                                                                                                                      │
│ - [12] C. Cui, T. Sun, S. Liang, et al. Paddleocr-vl: Boosting multilingual document parsing via a 0.9 b ultra-compact vision-language model. arXiv preprint arXiv:2510.14528, 2025.                 │
│ - [13] C. Cui, T. Sun, M. Lin, T. Gao, Y. Zhang, J. Liu, X. Wang, Z. Zhang, C. Zhou, H. Liu, et al. Paddleocr 3.0 technical report. arXiv preprint arXiv:2507.05595, 2025.                           │
│ - [14] M. Dehghani, J. Djolonga, B. Mustafa, P. Padlewski, J. Heek, J. Gilmer, A. Steiner, M. Caron, R. Geirhos, I. Alabdulmohsin, et al. Patch n' pack: Navit, a vision transforme

╭─ chunk_pos=57 num_tokens=1090 doc_items_refs=['#/texts/232', '#/texts/233', '#/texts/234', '#/texts/235', '#/texts/236', '#/texts/237', '#/texts/238', '#/texts/239', '#/texts/240', '#/texts/241', ─╮
│ References                                                                                                                                                                                           │
│ - [18] K. He, X. Zhang, S. Ren, and J. Sun. Deep residual learning for image recognition. In Proceedings of the IEEE conference on computer vision and pattern recognition, pages 770-778, 2016.     │
│ - [19] A. Huang, C. Yao, C. Han, F. Wan, H. Guo, H. Lv, H. Zhou, J. Wang, J. Zhou, J. Sun, et al. Step3-vl-10b technical report. arXiv preprint arXiv:2601.09668, 2026.                              │
│ - [20] S. Iyer, X. V . Lin, R. Pasunuru, T. Mihaylov, D. Simig, P. Yu, K. Shuster, T. Wang, Q. Liu, P. S. Koura, et al. Opt-iml: Scaling language model instruction meta learning through the lens   │
│ of generalization. arXiv preprint arXiv:2212.12017, 2022.                                                                                                                                            │
│ - [21] A. Kirillov, E. Mintun, N. Ravi, H. Mao, C. Rolland, L. Gustafson, T. Xiao, S. Whitehead, A. C. Berg, W.-Y. Lo, et al. Segment anything. arXiv preprint arXiv:2304.02643, 2023.               │
│ - [22] J. Li, D. Li, S. Savarese, and S. Hoi. Blip-2: Bootstrapping language-image pre-training with frozen image encoders and large language models. In International conference on machine         │
│ learning, pages 19730-19742. PMLR, 2023.                                                                                                                                                             │
│ - [23] Z. Li, Y. Liu, Q. Liu, Z. Ma, Z. Zhang, S. Zhang, Z. Guo, J. Zhang, X. Wang, and X. Bai. Monkeyocr: Document parsing with a structure-recognition-relation triplet paradigm. arXiv preprint   │
│ arXiv:2506.05218, 2025.                                                                                                                                                                              │
│ - [24] A. Liu, B. Feng, B. Wang, B. Wang, B. Liu, C. Zhao, C. Dengr, C. Ruan, D. Dai, D. Guo, et al. Deepseek-v2: A strong, economical, and efficient mixture-of-experts language model. arXiv       │
│ preprint arXiv:2405.04434, 2024.                                                                                                                                                                     │
│ - [25] A. Liu, B. Feng, B. Xue, B. Wang, B. Wu, C. Lu, C. Zhao, C. Deng, C. Zhang, C. Ruan, et al. Deepseek-v3 technical report. arXiv preprint arXiv:2412.19437, 2024.                              │
│ - [26] A. Liu, A. Mei, B. Lin, B. Xue, B. Wang, B. Xu, B. Wu, B. Zhang, C. Lin, C. Dong, et al. Deepseek-v3. 2: Pushing the frontier of open large language models. arXiv preprint arXiv:2512.02556, │
│ 2025.                                                                                                                                                                                                │
│ - [27] C. Liu, H. Wei, J. Chen, L. Kong, Z. Ge, Z. Zhu, L. Zhao, J. Sun, C. Han, and X. Zhang. Focus anywhere for fine-grained multi-page document understanding. arXiv preprint arXiv:2405.14295,   │
│ 2024.                                                                                                                                                                                                │
│ - [28] F. Liu and H. Qiu. Context cascade compression: Exploring the upper limits of text compression. arXiv preprint arXiv:2511.15244, 2025.                                                        │
│ - [29] F. Liu, H. Wei, W. Zhao, G. Li, J. Peng, and Z. Li. Wb-detr: Transformer-based detector without backbone. In Proceedings of the IEEE/CVF International Conference on Compute

╭─ chunk_pos=58 num_tokens=1033 doc_items_refs=['#/texts/249', '#/texts/250', '#/texts/251', '#/texts/252', '#/texts/253', '#/texts/254', '#/texts/255', '#/texts/256', '#/texts/257', '#/texts/258', ─╮
│ References                                                                                                                                                                                           │
│ - [34] L. Ouyang, Y. Qu, H. Zhou, J. Zhu, R. Zhang, Q. Lin, B. Wang, Z. Zhao, M. Jiang, X. Zhao, et al. Omnidocbench: Benchmarking diverse pdf document parsing with comprehensive annotations. In   │
│ Proceedings of the Computer Vision and Pattern Recognition Conference, pages 24838-24848, 2025.                                                                                                      │
│ - [35] Z. Pang, Z. Xie, Y. Man, and Y.-X. Wang. Frozen transformers in language models are effective visual encoder layers. arXiv preprint arXiv:2310.12973, 2023.                                   │
│ - [36] J. Poznanski, A. Rangapur, J. Borchardt, J. Dunkelberger, R. Huff, D. Lin, C. Wilhelm, K. Lo, and L. Soldaini. olmocr: Unlocking trillions of tokens in pdfs with vision language models.     │
│ arXiv preprint arXiv:2502.18443, 2025.                                                                                                                                                               │
│ - [37] A. Radford, J. W. Kim, C. Hallacy, A. Ramesh, G. Goh, S. Agarwal, G. Sastry, A. Askell, P. Mishkin, J. Clark, et al. Learning transferable visual models from natural language supervision.   │
│ In International conference on machine learning, pages 8748-8763. PMLR, 2021.                                                                                                                        │
│ - [38] J. Redmon and A. Farhadi. Yolo9000: better, faster, stronger. In Proceedings of the IEEE conference on computer vision and pattern recognition, pages 7263-7271, 2017.                        │
│ - [39] Rednote. dots.ocr, 2025. URL https://github.com/rednote-hilab/dots.ocr .                                                                                                                      │
│ - [40] S. Ren, K. He, R. Girshick, and J. Sun. Faster r-cnn: Towards real-time object detection with region proposal networks. In Advances in neural information processing systems, pages 91-99,    │
│ 2015.                                                                                                                                                                                                │
│ - [41] B. Seed. Seed1. 8 model card: Towards generalized real-world agency, 2025.                                                                                                                    │
│ - [42] J. Su, Y. Lu, S. Pan, B. Wen, and Y. Liu. Roformer: Enhanced transformer with rotary position embedding. arXiv preprint arXiv:2104.09864, 2021.                                               │
│ - [43] C. Team. Chameleon: Mixed-modal early-fusion foundation models. arXiv preprint arXiv:2405.09818, 2024.                                                                                        │
│ - [44] G. Team, R. Anil, S. Borgeaud, J.-B. Alayrac, J. Yu, R. Soricut, J. Schalkwyk, A. M. Dai, A. Hauth, K. Millican, et al. Gemini: a family of highly capable multimodal models. arXiv preprint  │
│ arXiv:2312.11805, 2023.                                                                                                                                                                              │
│ - [45] B. Wang, C. Xu, X. Zhao, L. Ouyang, F. Wu, Z. Zhao, R. Xu, K. Liu, Y. Qu, F. Shang, et al. Mineru: An open-source solution for precise document content extraction. arXiv preprint            │
│ arXiv:2409.18839, 2024.                                                                                                                                                            

╭──────────────── chunk_pos=59 num_tokens=575 doc_items_refs=['#/texts/265', '#/texts/266', '#/texts/267', '#/texts/268', '#/texts/269', '#/texts/270', '#/texts/271', '#/texts/272'] ─────────────────╮
│ References                                                                                                                                                                                           │
│ - [49] W. Wang, Z. Gao, L. Gu, et al. Internvl3.5: Advancing open-source multimodal models in versatility, reasoning, and efficiency. arXiv preprint arXiv:2508.18265, 2025.                         │
│ - [50] H. Wei, L. Kong, J. Chen, L. Zhao, Z. Ge, J. Yang, J. Sun, C. Han, and X. Zhang. Vary: Scaling up the vision vocabulary for large vision-language model. In European Conference on Computer   │
│ Vision, pages 408-424. Springer, 2024.                                                                                                                                                               │
│ - [51] H. Wei, L. Kong, J. Chen, L. Zhao, Z. Ge, E. Yu, J. Sun, C. Han, and X. Zhang. Small language model meets with reinforced vision vocabulary. arXiv preprint arXiv:2401.12503, 2024.           │
│ - [52] H. Wei, C. Liu, J. Chen, J. Wang, L. Kong, Y. Xu, Z. Ge, L. Zhao, J. Sun, Y. Peng, et al. General ocr theory: Towards ocr-2.0 via a unified end-to-end model. arXiv preprint                  │
│ arXiv:2409.01704, 2024.                                                                                                                                                                              │
│ - [53] H. Wei, Y. Yin, Y. Li, J. Wang, L. Zhao, J. Sun, Z. Ge, X. Zhang, and D. Jiang. Slow perception: Let's perceive geometric figures step-by-step. arXiv preprint arXiv:2412.20631, 2024.        │
│ - [54] H. Wei, Y. Sun, and Y. Li. Deepseek-ocr: Contexts optical compression. arXiv preprint arXiv:2510.18234, 2025.                                                                                 │
│ - [55] J. Zhu, W. Wang, Z. Chen, Z. Liu, S. Ye, L. Gu, H. Tian, Y. Duan, W. Su, J. Shao, et al. Internvl3: Exploring advanced training and test-time recipes for open-source multimodal models.      │
│ arXiv preprint arXiv:2504.10479, 2025.                                                                                                                                                               │
│ - [56] X. Zhu, W. Su, L. Lu, B. Li, X. Wang, and J. Dai. Deformable detr: Deformable transformers for end-to-end object detection. arXiv preprint arXiv:2010.04159, 2020.                            │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
from docling_core.transforms.serializer.markdown import MarkdownParams


class ImgPlaceholderSerializerProvider(ChunkingSerializerProvider):
    def get_serializer(self, doc):
        return ChunkingDocSerializer(
            doc=doc,
            params=MarkdownParams(
                image_placeholder="<!-- image -->",
            ),
        )


chunker = HierarchicalChunker(
    serializer_provider=ImgPlaceholderSerializerProvider(),
    merge_list_items=True,
)

chunk_iter = chunker.chunk(dl_doc=document)

chunks = list(chunk_iter)
i, chunk = find_n_th_chunk_with_label(chunks, n=0, label=DocItemLabel.PICTURE)
print_chunk(
    chunks=chunks,
    chunk_pos=i,
)

╭────────────────────────────────────────────────────────────── chunk_pos=29 num_tokens=25 doc_items_refs=['#/texts/34', '#/pictures/0'] ──────────────────────────────────────────────────────────────╮
│ Related Work                                                                                                                                                                                         │
│ Figure 1: System architecture: Simplified sketch of document question-answering pipeline.                                                                                                            │
│                                                                                                                                                                                                      │
│ <!-- image -->                                                                                                                                                                                       │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯